# Pong DQN en Google Colab

Antes de correr: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU (T4)**.

Dos formas de usarlo:

- **Solo ver el agente ya entrenado (unos 3 minutos):** celdas 1 y A. No hace falta Drive.
- **Entrenar desde cero (unas 2 horas):** celdas 1 a 5. Los resultados y los pesos se
  guardan en Google Drive (`MyDrive/pong_dqn_v2/`), así que si Colab se desconecta no se
  pierde nada y se puede retomar con `--reanudar`.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 1. Código y dependencias

In [ ]:
!git clone -q https://github.com/glizano/pong_dqn.git /content/pong_dqn || (cd /content/pong_dqn && git pull -q)
# Colab trae torch con CUDA; solo faltan el emulador de Atari y opencv.
!pip install -q "gymnasium[atari]>=1.2.3" opencv-python-headless

## A. Ver el agente entrenado (opcional, sin entrenar)

Carga el mejor modelo que viene en el repositorio (`saves/dqn_v2/mejor.pt`), lo evalúa en
10 partidos y graba un GIF de una partida junto a lo que ve la red.

In [ ]:
from IPython.display import Image

%cd /content/pong_dqn
!PYTHONPATH=src python -m pong_dqn.cli evaluar --modelo saves/dqn_v2/mejor.pt --n 10
!PYTHONPATH=src python scripts/grabar_gif.py --modelo saves/dqn_v2/mejor.pt --pasos 700

Image(filename='resultados/partida.gif')

## 2. Carpeta de trabajo en Drive

In [ ]:
import os

from google.colab import drive

drive.mount('/content/drive')


TRABAJO = '/content/drive/MyDrive/pong_dqn_v2'
os.makedirs(TRABAJO, exist_ok=True)
os.chdir(TRABAJO)  # resultados/ y saves/ se crean aqui
os.environ['PYTHONPATH'] = '/content/pong_dqn/src'
print(os.getcwd())

## 3. Prueba de velocidad (30 s)

Da los pasos por segundo con esta GPU y cuánto tardaría el millón de pasos.

In [ ]:
!python -m pong_dqn.cli benchmark --segundos 30

## 4. Entrenamiento

`--etiqueta` separa esta corrida de la del Mac. Si Colab se desconecta, vuelva a correr las
celdas 1 y 2 y esta misma celda agregando `--reanudar`.

In [ ]:
!python -m pong_dqn.cli entrenar --pasos 1000000 --etiqueta dqn_v2 --semilla 1

## 5. Evaluación final y gráfica

In [ ]:
!python -m pong_dqn.cli evaluar --modelo saves/dqn_v2/mejor.pt --n 30 --salida resultados/dqn_v2/final_sticky0.json
!python -m pong_dqn.cli evaluar --modelo saves/dqn_v2/mejor.pt --n 30 --sticky 0.25 --salida resultados/dqn_v2/final_sticky025.json
!cd {TRABAJO} && python /content/pong_dqn/scripts/graficar.py dqn_v2